In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "customer_support_tickets.csv"
)

df = pd.read_csv(RAW_DATA_FILE)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [2]:
# Check the target labels and input text quality

text_columns = ["Ticket Subject", "Ticket Description"]
target_column = "Ticket Type"

print("Missing target labels:", df[target_column].isna().sum())

for column in text_columns:
    print(f"\n{column}")
    print("Missing:", df[column].isna().sum())
    print("Empty:", df[column].fillna("").str.strip().eq("").sum())
    print("Unique values:", df[column].nunique())

print("\nTarget distribution:")
display(df[target_column].value_counts(dropna=False).to_frame("count"))

Missing target labels: 0

Ticket Subject
Missing: 0
Empty: 0
Unique values: 16

Ticket Description
Missing: 0
Empty: 0
Unique values: 8077

Target distribution:


,count
Ticket Type,
Refund request,1752
Technical issue,1747
Cancellation request,1695
Product inquiry,1641
Billing inquiry,1634


In [3]:
from IPython.display import display

subject_type_table = pd.crosstab(
    df["Ticket Subject"],
    df["Ticket Type"],
    normalize="index"
).round(2)

display(subject_type_table)

Ticket Type,Billing inquiry,Cancellation request,Product inquiry,Refund request,Technical issue
Ticket Subject,,,,,
Account access,0.20,0.18,0.21,0.21,0.19
Battery life,0.20,0.19,0.19,0.22,0.21
Cancellation request,0.17,0.21,0.17,0.22,0.22
Data loss,0.18,0.23,0.19,0.20,0.20
Delivery problem,0.20,0.20,0.19,0.19,0.21
Display issue,0.19,0.22,0.17,0.21,0.22
Hardware issue,0.18,0.20,0.19,0.24,0.19
Installation support,0.20,0.19,0.19,0.22,0.20
Network problem,0.18,0.19,0.21,0.20,0.23


In [4]:
# Check duplicate descriptions and whether their labels agree

description_stats = (
    df.groupby("Ticket Description")["Ticket Type"]
    .agg(["count", "nunique"])
    .sort_values("count", ascending=False)
)

print("Unique descriptions:", df["Ticket Description"].nunique())
print("Repeated descriptions:", (description_stats["count"] > 1).sum())

print(
    "Repeated descriptions with conflicting labels:",
    (
        (description_stats["count"] > 1)
        & (description_stats["nunique"] > 1)
    ).sum()
)

display(description_stats.head(10))

Unique descriptions: 8077
Repeated descriptions: 47
Repeated descriptions with conflicting labels: 42


,count,nunique
Ticket Description,,
I'm having an issue with the {product_purchased}. Please assist. I need assistance as soon as possible because it's affecting my work and productivity.,25,5
I'm having an issue with the {product_purchased}. Please assist. I'm concerned about the security of my {product_purchased} and would like to ensure that my data is safe.,25,5
I'm having an issue with the {product_purchased}. Please assist. This problem started occurring after the recent software update. I haven't made any other changes to the device.,25,5
"I'm having an issue with the {product_purchased}. Please assist. I've already contacted customer support multiple times, but the issue remains unresolved.",24,5
I'm having an issue with the {product_purchased}. Please assist. I've noticed a peculiar error message popping up on my {product_purchased} screen. It says '{error_message}'. What does it mean?,24,5
"I'm having an issue with the {product_purchased}. Please assist. I've performed a factory reset on my {product_purchased}, hoping it would resolve the problem, but it didn't help.",23,5
"I'm having an issue with the {product_purchased}. Please assist. The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.",20,5
I'm having an issue with the {product_purchased}. Please assist. I'm unable to find the option to perform the desired action in the {product_purchased}. Could you please guide me through the steps?,20,5
I'm having an issue with the {product_purchased}. Please assist. I've noticed that the issue occurs consistently when I use a specific feature or application on my {product_purchased}.,17,4
